# 01 - Data Exploration

Notebook nay kiem tra nhanh bo du lieu da xu ly trong `data/processed`:
- Kich thuoc file + so dong
- Phan phoi rating
- So review / user / item theo tung category

In [1]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REVIEWS_FILE = PROCESSED_DIR / 'reviews_clean.jsonl'
INTERACTIONS_FILE = PROCESSED_DIR / 'interactions.jsonl'
META_FILE = PROCESSED_DIR / 'products_meta.jsonl'

for p in [REVIEWS_FILE, INTERACTIONS_FILE, META_FILE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: c:\Users\bonco\SentimentAnalysisApp
PROCESSED_DIR: c:\Users\bonco\SentimentAnalysisApp\data\processed


In [2]:
def iter_jsonl(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)


def count_rows(path: Path) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

In [3]:
# 1) Kich thuoc va quy mo du lieu
files = {
    'reviews_clean': REVIEWS_FILE,
    'interactions': INTERACTIONS_FILE,
    'products_meta': META_FILE,
}

size_rows = []
for name, path in files.items():
    size_rows.append({
        'dataset': name,
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_mb': round(file_size_mb(path), 2),
        'rows': count_rows(path),
    })

if pd is not None:
    size_df = pd.DataFrame(size_rows).sort_values('dataset').reset_index(drop=True)
    display(size_df)
else:
    print(size_rows)

[{'dataset': 'reviews_clean', 'path': 'data\\processed\\reviews_clean.jsonl', 'size_mb': 1278.0, 'rows': 1689188}, {'dataset': 'interactions', 'path': 'data\\processed\\interactions.jsonl', 'size_mb': 155.84, 'rows': 1689188}, {'dataset': 'products_meta', 'path': 'data\\processed\\products_meta.jsonl', 'size_mb': 129.77, 'rows': 498196}]


In [4]:
# 2) Phan phoi rating
rating_counter = Counter()
total_reviews = 0

for row in iter_jsonl(REVIEWS_FILE):
    rating = row.get('rating')
    if rating is None:
        continue
    rating = float(rating)
    rating_counter[rating] += 1
    total_reviews += 1

rating_rows = []
for r in sorted(rating_counter):
    c = rating_counter[r]
    rating_rows.append({
        'rating': r,
        'count': c,
        'ratio_pct': round(c * 100.0 / total_reviews, 2),
    })

if pd is not None:
    rating_df = pd.DataFrame(rating_rows)
    display(rating_df)
else:
    print(rating_rows)

if plt is not None and rating_rows:
    x = [str(r['rating']) for r in rating_rows]
    y = [r['count'] for r in rating_rows]
    plt.figure(figsize=(8, 4))
    plt.bar(x, y)
    plt.title('Rating Distribution')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

[{'rating': 1.0, 'count': 108725, 'ratio_pct': 6.44}, {'rating': 2.0, 'count': 82139, 'ratio_pct': 4.86}, {'rating': 3.0, 'count': 142257, 'ratio_pct': 8.42}, {'rating': 4.0, 'count': 347041, 'ratio_pct': 20.54}, {'rating': 5.0, 'count': 1009026, 'ratio_pct': 59.73}]


In [5]:
# 3) So review / user / item moi category
asin_to_category = {}

for row in iter_jsonl(META_FILE):
    asin = row.get('asin')
    if not asin:
        continue

    category = row.get('category_leaf')
    if not category:
        category_path = row.get('category_path') or []
        category = category_path[-1] if category_path else 'Unknown'

    asin_to_category[asin] = category or 'Unknown'

print(f'Loaded category mapping for {len(asin_to_category):,} items')

category_review_count = Counter()
category_users = defaultdict(set)
category_items = defaultdict(set)

for idx, row in enumerate(iter_jsonl(REVIEWS_FILE), start=1):
    asin = row.get('asin')
    user_id = row.get('reviewer_id')

    category = asin_to_category.get(asin, 'Unknown')
    category_review_count[category] += 1

    if user_id:
        category_users[category].add(user_id)
    if asin:
        category_items[category].add(asin)

    if idx % 300000 == 0:
        print(f'Processed {idx:,} reviews...')

category_rows = []
for category, review_count in category_review_count.items():
    user_count = len(category_users[category])
    item_count = len(category_items[category])
    category_rows.append({
        'category': category,
        'review_count': review_count,
        'user_count': user_count,
        'item_count': item_count,
        'reviews_per_user': round(review_count / user_count, 3) if user_count else 0.0,
        'reviews_per_item': round(review_count / item_count, 3) if item_count else 0.0,
    })

if pd is not None:
    category_df = pd.DataFrame(category_rows).sort_values('review_count', ascending=False).reset_index(drop=True)
    display(category_df.head(30))
else:
    category_rows = sorted(category_rows, key=lambda x: x['review_count'], reverse=True)
    print(category_rows[:30])

Loaded category mapping for 498,196 items
Processed 300,000 reviews...
Processed 600,000 reviews...
Processed 900,000 reviews...
Processed 1,200,000 reviews...
Processed 1,500,000 reviews...
[{'category': 'Cases', 'review_count': 95128, 'user_count': 60236, 'item_count': 3505, 'reviews_per_user': 1.579, 'reviews_per_item': 27.141}, {'category': 'Headphones', 'review_count': 68706, 'user_count': 44799, 'item_count': 2135, 'reviews_per_user': 1.534, 'reviews_per_item': 32.181}, {'category': 'HDMI Cables', 'review_count': 40883, 'user_count': 31985, 'item_count': 713, 'reviews_per_user': 1.278, 'reviews_per_item': 57.339}, {'category': 'Point & Shoot Digital Cameras', 'review_count': 29373, 'user_count': 22195, 'item_count': 1146, 'reviews_per_user': 1.323, 'reviews_per_item': 25.631}, {'category': 'External Hard Drives', 'review_count': 28346, 'user_count': 22200, 'item_count': 596, 'reviews_per_user': 1.277, 'reviews_per_item': 47.56}, {'category': 'Mice', 'review_count': 27116, 'user_c

In [6]:
# Luu ket qua thong ke theo category de dung lai
output_csv = PROCESSED_DIR / 'category_stats.csv'

if pd is not None:
    category_df.to_csv(output_csv, index=False)
else:
    fieldnames = [
        'category',
        'review_count',
        'user_count',
        'item_count',
        'reviews_per_user',
        'reviews_per_item',
    ]
    with open(output_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in category_rows:
            writer.writerow(row)

print('Saved:', output_csv)

Saved: c:\Users\bonco\SentimentAnalysisApp\data\processed\category_stats.csv


## 4) Sentiment Bias Theo Category Va User
Phan nay tap trung tra loi cau hoi: sentiment co bi thien lech (bias) theo tung category va tung user hay khong.

Chi so su dung:
- Category bias: do lech ti le Positive/Negative cua moi category so voi toan bo dataset.
- User bias: muc do mot user danh gia nghieng ve 1 nhan (dominant sentiment ratio).
- Ket luan dua tren nguong tham khao va so mau toi thieu de tranh nhieu do noise.

In [ ]:
# 4.1) Tinh toan bias theo category va user
from collections import Counter, defaultdict
import csv
import statistics
from pathlib import Path

CATEGORY_DIST_FILE = PROCESSED_DIR / 'category_sentiment_distribution.csv'
if not CATEGORY_DIST_FILE.exists():
    raise FileNotFoundError(f'Missing required file: {CATEGORY_DIST_FILE}')

# ---- Category-level bias ----
category_dist_rows = []
with open(CATEGORY_DIST_FILE, 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        category_dist_rows.append({
            'category': row['category'],
            'total_reviews': int(row['total_reviews']),
            'negative_count': int(row['negative_count']),
            'neutral_count': int(row['neutral_count']),
            'positive_count': int(row['positive_count']),
            'negative_ratio_pct': float(row['negative_ratio_pct']),
            'neutral_ratio_pct': float(row['neutral_ratio_pct']),
            'positive_ratio_pct': float(row['positive_ratio_pct']),
        })

total_reviews_all = sum(r['total_reviews'] for r in category_dist_rows)
total_negative_all = sum(r['negative_count'] for r in category_dist_rows)
total_neutral_all = sum(r['neutral_count'] for r in category_dist_rows)
total_positive_all = sum(r['positive_count'] for r in category_dist_rows)

global_negative_pct = (total_negative_all / total_reviews_all) * 100 if total_reviews_all else 0.0
global_neutral_pct = (total_neutral_all / total_reviews_all) * 100 if total_reviews_all else 0.0
global_positive_pct = (total_positive_all / total_reviews_all) * 100 if total_reviews_all else 0.0

MIN_CATEGORY_REVIEWS = 1000
category_bias_rows = []
for row in category_dist_rows:
    if row['total_reviews'] < MIN_CATEGORY_REVIEWS:
        continue

    delta_pos = row['positive_ratio_pct'] - global_positive_pct
    delta_neg = row['negative_ratio_pct'] - global_negative_pct
    delta_neu = row['neutral_ratio_pct'] - global_neutral_pct
    bias_score = max(abs(delta_pos), abs(delta_neg), abs(delta_neu))

    category_bias_rows.append({
        'category': row['category'],
        'total_reviews': row['total_reviews'],
        'positive_ratio_pct': round(row['positive_ratio_pct'], 4),
        'negative_ratio_pct': round(row['negative_ratio_pct'], 4),
        'neutral_ratio_pct': round(row['neutral_ratio_pct'], 4),
        'delta_positive_pct': round(delta_pos, 4),
        'delta_negative_pct': round(delta_neg, 4),
        'delta_neutral_pct': round(delta_neu, 4),
        'bias_score_pct': round(bias_score, 4),
    })

category_bias_rows.sort(key=lambda x: x['bias_score_pct'], reverse=True)
top_category_bias = category_bias_rows[:20]

print('GLOBAL SENTIMENT DISTRIBUTION (%)')
print({
    'negative_pct': round(global_negative_pct, 4),
    'neutral_pct': round(global_neutral_pct, 4),
    'positive_pct': round(global_positive_pct, 4),
})
print(f'Categories considered (>= {MIN_CATEGORY_REVIEWS} reviews): {len(category_bias_rows)}')

if pd is not None:
    category_bias_df = pd.DataFrame(top_category_bias)
    display(category_bias_df)
else:
    category_bias_df = None
    print(top_category_bias[:10])

# ---- User-level bias ----
user_sentiment_counts = defaultdict(Counter)
for row in iter_jsonl(REVIEWS_FILE):
    user_id = row.get('reviewer_id')
    sentiment = row.get('sentiment')
    if not user_id or sentiment not in {'Negative', 'Neutral', 'Positive'}:
        continue
    user_sentiment_counts[user_id][sentiment] += 1

USER_MIN_REVIEWS = 5
user_profiles = []
for user_id, cnt in user_sentiment_counts.items():
    total = cnt['Negative'] + cnt['Neutral'] + cnt['Positive']
    if total < USER_MIN_REVIEWS:
        continue

    dominant_sentiment = max(['Negative', 'Neutral', 'Positive'], key=lambda s: cnt[s])
    dominant_ratio = cnt[dominant_sentiment] / total
    user_profiles.append({
        'user_id': user_id,
        'total_reviews': total,
        'negative': cnt['Negative'],
        'neutral': cnt['Neutral'],
        'positive': cnt['Positive'],
        'dominant_sentiment': dominant_sentiment,
        'dominant_ratio': dominant_ratio,
    })

num_users_total = len(user_sentiment_counts)
num_users_profiled = len(user_profiles)
dominant_ratio_values = [u['dominant_ratio'] for u in user_profiles]
dominant_ge_80 = sum(1 for r in dominant_ratio_values if r >= 0.8)
dominant_ge_90 = sum(1 for r in dominant_ratio_values if r >= 0.9)

heavy_threshold = 20
heavy_users = [u for u in user_profiles if u['total_reviews'] >= heavy_threshold]
light_users = [u for u in user_profiles if USER_MIN_REVIEWS <= u['total_reviews'] < heavy_threshold]

def agg_group(users):
    neg = sum(u['negative'] for u in users)
    neu = sum(u['neutral'] for u in users)
    pos = sum(u['positive'] for u in users)
    total = neg + neu + pos
    if total == 0:
        return {'negative_pct': 0.0, 'neutral_pct': 0.0, 'positive_pct': 0.0, 'total_reviews': 0}
    return {
        'negative_pct': round(neg * 100 / total, 4),
        'neutral_pct': round(neu * 100 / total, 4),
        'positive_pct': round(pos * 100 / total, 4),
        'total_reviews': total,
    }

heavy_dist = agg_group(heavy_users)
light_dist = agg_group(light_users)

dominant_sentiment_counter = Counter(u['dominant_sentiment'] for u in user_profiles)
user_bias_summary = {
    'users_total': num_users_total,
    'users_profiled_min_reviews': num_users_profiled,
    'dominant_ge_80_ratio_pct': round((dominant_ge_80 * 100 / num_users_profiled), 4) if num_users_profiled else 0.0,
    'dominant_ge_90_ratio_pct': round((dominant_ge_90 * 100 / num_users_profiled), 4) if num_users_profiled else 0.0,
    'median_dominant_ratio': round(statistics.median(dominant_ratio_values), 4) if dominant_ratio_values else 0.0,
    'dominant_sentiment_distribution': dict(dominant_sentiment_counter),
    'heavy_user_count': len(heavy_users),
    'light_user_count': len(light_users),
    'heavy_user_sentiment_pct': heavy_dist,
    'light_user_sentiment_pct': light_dist,
}

print('\nUSER BIAS SUMMARY')
print(user_bias_summary)

if pd is not None and user_profiles:
    top_user_bias_df = pd.DataFrame(
        sorted(user_profiles, key=lambda x: (x['dominant_ratio'], x['total_reviews']), reverse=True)[:20]
    )
    display(top_user_bias_df)
else:
    top_user_bias_df = None

# Rule-of-thumb ket luan bias
category_bias_flag = any(r['bias_score_pct'] >= 10.0 for r in category_bias_rows)
user_bias_flag = user_bias_summary['dominant_ge_80_ratio_pct'] >= 30.0
bias_conclusion = {
    'category_bias_detected': category_bias_flag,
    'user_bias_detected': user_bias_flag,
    'overall_bias_detected': bool(category_bias_flag or user_bias_flag),
}
print('\nBIAS CONCLUSION (RULE-OF-THUMB):', bias_conclusion)

GLOBAL SENTIMENT DISTRIBUTION (%)
{'negative_pct': 11.2992, 'neutral_pct': 8.4216, 'positive_pct': 80.2792}
Categories considered (>= 1000 reviews): 258
[{'category': 'Hidden Cameras', 'total_reviews': 1721, 'positive_ratio_pct': 60.7786, 'negative_ratio_pct': 24.6368, 'neutral_ratio_pct': 14.5845, 'delta_positive_pct': -19.5006, 'delta_negative_pct': 13.3376, 'delta_neutral_pct': 6.1629, 'bias_score_pct': 19.5006}, {'category': 'Bullet Cameras', 'total_reviews': 1398, 'positive_ratio_pct': 61.2303, 'negative_ratio_pct': 23.8913, 'neutral_ratio_pct': 14.8784, 'delta_positive_pct': -19.0489, 'delta_negative_pct': 12.5921, 'delta_neutral_pct': 6.4568, 'bias_score_pct': 19.0489}, {'category': 'FM Transmitters', 'total_reviews': 6258, 'positive_ratio_pct': 62.1924, 'negative_ratio_pct': 25.4554, 'neutral_ratio_pct': 12.3522, 'delta_positive_pct': -18.0868, 'delta_negative_pct': 14.1562, 'delta_neutral_pct': 3.9306, 'bias_score_pct': 18.0868}, {'category': 'Network Antennas', 'total_reviews

: 

In [8]:
# 4.2) Sinh doan van viet san cho luan van Chuong 3
chapter3_text = f'''### 3.x. Phan tich thien lech sentiment theo category va user
Trong bo du lieu nay, phan phoi sentiment toan cuc la: Positive = {global_positive_pct:.2f}%, Neutral = {global_neutral_pct:.2f}%, Negative = {global_negative_pct:.2f}%.
Voi nguong toi thieu {MIN_CATEGORY_REVIEWS} review/category, co {len(category_bias_rows)} category duoc dua vao phan tich.

Ket qua cho thay do lech sentiment theo category co ton tai o mot so nhom san pham: category co bias_score cao nhat dat {category_bias_rows[0]['bias_score_pct']:.2f} diem phan tram (neu co du lieu). Dieu nay ham y sentiment phu thuoc dang ke vao dac thu category.

O cap do user, trong so {num_users_profiled:,} user co it nhat {USER_MIN_REVIEWS} review, ty le user co nhan chiem uu the >= 80% la {user_bias_summary['dominant_ge_80_ratio_pct']:.2f}%, va >= 90% la {user_bias_summary['dominant_ge_90_ratio_pct']:.2f}%.
Median dominant ratio dat {user_bias_summary['median_dominant_ratio']:.2f}, cho thay xu huong danh gia lech ve mot nhan cua tung nhom user.

Tu cac chi bao tren, ket luan quy tac (rule-of-thumb) cho thay: category_bias_detected={bias_conclusion['category_bias_detected']}, user_bias_detected={bias_conclusion['user_bias_detected']}, overall_bias_detected={bias_conclusion['overall_bias_detected']}.

Ham y thuc nghiem: khi huan luyen mo hinh phan loai sentiment, can xem xet xu ly mat can bang lop (class_weight/SMOTE), stratified split, va danh gia them theo tung category de tranh overfit vao nhom Positive chiem uu the.'''

print(chapter3_text)

chapter3_out = PROCESSED_DIR / 'chapter3_bias_section.md'
with open(chapter3_out, 'w', encoding='utf-8') as f:
    f.write(chapter3_text)

print('Saved thesis section draft to:', chapter3_out)

### 3.x. Phan tich thien lech sentiment theo category va user
Trong bo du lieu nay, phan phoi sentiment toan cuc la: Positive = 80.28%, Neutral = 8.42%, Negative = 11.30%.
Voi nguong toi thieu 1000 review/category, co 258 category duoc dua vao phan tich.

Ket qua cho thay do lech sentiment theo category co ton tai o mot so nhom san pham: category co bias_score cao nhat dat 19.50 diem phan tram (neu co du lieu). Dieu nay ham y sentiment phu thuoc dang ke vao dac thu category.

O cap do user, trong so 192,401 user co it nhat 5 review, ty le user co nhan chiem uu the >= 80% la 65.53%, va >= 90% la 36.61%.
Median dominant ratio dat 0.83, cho thay xu huong danh gia lech ve mot nhan cua tung nhom user.

Tu cac chi bao tren, ket luan quy tac (rule-of-thumb) cho thay: category_bias_detected=True, user_bias_detected=True, overall_bias_detected=True.

Ham y thuc nghiem: khi huan luyen mo hinh phan loai sentiment, can xem xet xu ly mat can bang lop (class_weight/SMOTE), stratified split, va danh 